# Session 6 — Recursion & Recursive Thinking

> base case + recursive case, the call stack, recursion vs iteration, recursing nested data.

**How to use this notebook:** read each cell, **predict** what it prints, then run it with **Shift + Enter**. Change one thing and predict again — the surprise is the lesson. Practice tasks (with collapsed solutions) are at the bottom.

In [ ]:
import sys
import functools

In [ ]:
# 1) The shape of EVERY recursive function: a base case + a recursive case.
def countdown(n):
    if n <= 0:                 # BASE CASE — the stop condition
        print("liftoff!")
        return
    print(n)
    countdown(n - 1)           # RECURSIVE CASE — same problem, smaller input

print("countdown:")
countdown(3)

In [ ]:
# 2) Recursion vs iteration: same answer, two styles. Each call adds a stack frame.
def factorial(n):
    if n <= 1:
        return 1               # base case
    return n * factorial(n - 1)   # remember to RETURN the recursive call

def factorial_loop(n):
    total = 1
    for k in range(2, n + 1):
        total *= k
    return total

print("\nfactorial(5):", factorial(5), "==", factorial_loop(5))

In [ ]:
# 3) Memoization: @lru_cache remembers past calls so each input is computed once.
#    Naive fibonacci recomputes the same values exponentially; the cache makes it linear.
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

@functools.lru_cache(maxsize=None)     # one decorator turns the slow version fast
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)

print("\nfib_naive(30):", fib_naive(30), "in", calls, "calls")
print("fib_fast(30): ", fib_fast(30), "->", fib_fast.cache_info())   # far fewer hits

In [ ]:
# 4) Mutual recursion: two functions that call each other toward a shared base case.
def is_even(n):
    return True if n == 0 else is_odd(n - 1)
def is_odd(n):
    return False if n == 0 else is_even(n - 1)
print("\nis_even(10):", is_even(10), "| is_odd(7):", is_odd(7))

In [ ]:
# 5) Where recursion SHINES: naturally NESTED data, where one loop can't reach
#    all the way down. This is shaped like a real nested-JSON export.
export = {
    "cohort": "2026",
    "students": [
        {"name": "Ana", "scores": [91, 88]},
        {"name": "Ben", "scores": [58, [60, 64]]},   # arbitrarily nested
    ],
}

def deep_sum(obj):
    """Add up every number found anywhere inside nested lists/dicts."""
    if isinstance(obj, bool):                 # bool is an int subclass (Session 2!)
        return 0
    if isinstance(obj, (int, float)):
        return obj
    if isinstance(obj, dict):
        return sum(deep_sum(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(deep_sum(x) for x in obj)
    return 0                                  # strings, None, etc. contribute nothing

print("\ndeep_sum of nested export:", deep_sum(export))   # 91+88+58+60+64 = 361

In [ ]:
# 6) The trap: with no reachable base case, recursion never stops. Python has no
#    tail-call optimization, so it just piles up stack frames until it gives up.
print("\nPython's recursion limit:", sys.getrecursionlimit())

def runaway(n):
    return runaway(n + 1)        # BUG: never reaches a base case

try:
    runaway(0)
except RecursionError:
    print("RecursionError: maximum recursion depth exceeded (as expected)")

## Now you try — practice

# Session 6 — Practice (60 min): Recursion & Recursive Thinking

## Task 1 — Recursive sum
Write `rsum(n)` that adds `1 + 2 + ... + n` **with recursion** (no loop).
Name the base case out loud before you write it. Test `rsum(5)` and `rsum(0)`.

## Task 2 — Recursion vs iteration
Write `reverse(s)` that reverses a string recursively. Then write the loop version.
Which reads more clearly to you? Test `reverse("data")`.

## Task 3 — Flatten nested data
Write `flatten(xs)` that turns a list-of-lists (nested to any depth) into one flat list:
`flatten([1, [2, [3, 4]], 5])` → `[1, 2, 3, 4, 5]`. This is the move for nested JSON/exports.

## Task 4 — How deep does it go?
Write `depth(xs)` returning how deeply a list is nested:
`depth([1, [2, [3, [4]]]])` → `4`, `depth([1, 2, 3])` → `1`, `depth(5)` → `0`.

## Task 5 — Trap check
1. Why does this raise `RecursionError`, and what's the fix?
   ```python
   def f(n):
       return n + f(n - 1)
   ```
2. This returns `None` instead of a number — why?
   ```python
   def fact(n):
       if n <= 1:
           return 1
       n * fact(n - 1)
   ```
3. Name one case where a plain loop is the better choice over recursion.

## Bonus — Pythonic idiom drill
One decorator makes exponential recursion instant by remembering past calls.

```python
import functools

@functools.cache                     # memoize: each n is computed once
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(35))                       # -> 9227465   (try this WITHOUT @cache... then wait)
```

---

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

```python
# 1
def rsum(n):
    if n == 0:                      # base case
        return 0
    return n + rsum(n - 1)
print(rsum(5), rsum(0))             # 15 0

# 2
def reverse(s):
    if s == "":                     # base case: empty string
        return ""
    return reverse(s[1:]) + s[0]    # all-but-first, reversed, then first
print(reverse("data"))             # "atad"
# loop version: "".join(reversed(s))  — usually clearer for flat strings

# 3
def flatten(xs):
    out = []
    for x in xs:
        if isinstance(x, list):
            out.extend(flatten(x))  # recurse into the sub-list
        else:
            out.append(x)
    return out
print(flatten([1, [2, [3, 4]], 5]))   # [1, 2, 3, 4, 5]

# 4
def depth(xs):
    if not isinstance(xs, list):
        return 0                              # a non-list has no nesting
    return 1 + max((depth(x) for x in xs), default=0)
print(depth([1, [2, [3, [4]]]]), depth([1, 2, 3]), depth(5))   # 4 1 0

# 5
# 1) No reachable base case -> the calls never stop -> stack overflows.
#    Fix: add `if n == 0: return 0` (or n <= 0) at the top.
# 2) The recursive case computes n*fact(n-1) but never RETURNs it,
#    so the function falls off the end and returns None. Add `return`.
# 3) A loop is better when the work is a simple flat sequence, or when the
#    depth could exceed ~1000 (Python has no tail-call optimization, so deep
#    recursion hits RecursionError where a loop would be fine).
```

</details>